In [109]:
from sympy.physics.quantum.operator import UnitaryOperator, Dagger
from cvxpy import Variable, Problem, Minimize, trace, real, kron
# from sympy.matrices import eye, zeros, ones
from numpy import matrix, eye, zeros, ones
from scipy.linalg import polar
from sympy import MatPow
from math import sqrt

In [101]:
n = 1
p = 0.01

rho = Variable((2**n, 2**n), hermitian=True)

In [102]:
C = eye(2**n)
Z = zeros((2**n, 2**n))
O = ones((2**n, 2**n))
U = UnitaryOperator('U')
# U*U._eval_inverse()
E = [sqrt(p)*C, sqrt(1-p)*(O-C)]
# E = [C]

sigma = matrix([[1.0, 0.0], [0.0, 0.0]])

expression = kron(E[j]@sigma@E[i], E[i]@rho@E[j])
for i in range(2):
    for j in range(2):
        if i == j and not i:
            continue
        expression += kron(E[j]@sigma@E[i], E[i]@rho@E[j])

expression

Expression(AFFINE, UNKNOWN, (4, 4))

In [108]:
constraints = [rho >> 0]
constraints += [trace(rho) == 1]

prob = Problem(Minimize(real(trace(expression))), constraints)
prob.solve()

rho0 = rho.value

In [110]:
polar(rho0)

(array([[1.+0.j, 0.+0.j],
        [0.+0.j, 1.+0.j]]),
 array([[0.50000002+0.j, 0.        +0.j],
        [0.        +0.j, 0.50000002+0.j]]))

In [70]:
from cvxpy.atoms.affine.promote import Promote

p = Promote(1, (2, 2))

expression.__dict__

{'_arg_groups': [Expression(CONSTANT, UNKNOWN, (2, 2)),
  Expression(AFFINE, UNKNOWN, (2, 2))],
 'id': 1676,
 'args': [Expression(CONSTANT, UNKNOWN, (2, 2)),
  Expression(AFFINE, UNKNOWN, (2, 2))],
 '_shape': (2, 2),
 'is_constant__cache__': {(): False, ('__dpp_scope_active__',): False},
 'is_affine__cache__': {(): True, ('__dpp_scope_active__',): True},
 'is_convex__cache__': {(): True, ('__dpp_scope_active__',): True},
 'is_concave__cache__': {(): True, ('__dpp_scope_active__',): True},
 'is_zero__cache__': {(): False},
 'is_nonneg__cache__': {(): False},
 'is_nonpos__cache__': {(): False}}

In [71]:
p.__dict__

{'promoted_shape': (2, 2),
 'id': 1909,
 'args': [Constant(CONSTANT, NONNEGATIVE, ())],
 '_shape': (2, 2)}

In [90]:
X = Variable((n+1,n+1), symmetric=True)

constraints = [X >> 0]
constraints += [X[0,0] == 1]
# for i in range(1,n+1):
#     for j in range(1,n+1):
#         if E[i-1,j-1] == 1:
#             constraints += [X[i,j] == 0]
for i in range(1,n+1):
    constraints += [X[0,i] == X[i,i]]
#     constraints += [X[i,i] == P_list[i-1]]

# a = zeros((n+1, n+1))
a = C@X

prob = Problem(Maximize(trace(a+a)-1), constraints)
prob.solve()

2.9999778509359434